<a href="https://colab.research.google.com/github/Warukunai/LLM/blob/main/Finetune_Chinese_Weibo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 利用中文微博評價資料進行Bert微調


In [19]:
! pip install transformers datasets
! pip install evaluate

## 下載微博評價資料

In [20]:
!wget https://github.com/shhuangmust/AI/raw/refs/heads/113-1/weibo_senti_100k.csv

--2025-04-14 17:32:33--  https://github.com/shhuangmust/AI/raw/refs/heads/113-1/weibo_senti_100k.csv
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shhuangmust/AI/refs/heads/113-1/weibo_senti_100k.csv [following]
--2025-04-14 17:32:33--  https://raw.githubusercontent.com/shhuangmust/AI/refs/heads/113-1/weibo_senti_100k.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19699818 (19M) [application/octet-stream]
Saving to: ‘weibo_senti_100k.csv.2’

weibo_senti_100k.cs 100%[===================>]  18.79M  --.-KB/s    in 0.1s    

2025-04-14 17:32:35 (170 MB/s) - ‘weibo_senti_100k.csv.2’ saved [196

## 讀取Weibo資料集
- 共有119988筆資料

In [21]:
from datasets import load_dataset, DatasetDict

ds = load_dataset("csv", data_files="weibo_senti_100k.csv")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 119988
    })
})


## 分割資料集
- 80%訓練(train)資料
- 10%測試(test)資料
- 10%驗證(valid)資料


In [22]:
train_testvalid = ds['train'].train_test_split(test_size=0.2)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']})


## 進行分詞

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-chinese")

def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/95990 [00:00<?, ? examples/s]

Map:   0%|          | 0/11999 [00:00<?, ? examples/s]

Map:   0%|          | 0/11999 [00:00<?, ? examples/s]

## 為簡化訓練，挑選10000筆作為訓練與測試資料

In [24]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(10000))
print(small_train_dataset)
print(small_eval_dataset)

Dataset({
    features: ['label', 'review', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})
Dataset({
    features: ['label', 'review', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})


## 列印一筆資料出來看

In [25]:
tokenized_datasets["train"][100]

{'label': 1,
 'review': '[鼓掌][鼓掌][鼓掌]//@云南创网科技有限公司:所有参会客户，转发微博原文，凭借邀请函，到答谢会现场微博区领取。@大元昌茶业官方微博 @51普洱网 @云南滇吉会议服务有限公司 @云游旅行网 @昆明好人缘家政 @昆明苏菲雅婚纱时尚摄影 @品摄影梦工厂 @丫丫手机网 @海马汽车昆明合达4S店',
 'input_ids': [101,
  138,
  7961,
  2958,
  140,
  138,
  7961,
  2958,
  140,
  138,
  7961,
  2958,
  140,
  120,
  120,
  137,
  756,
  1298,
  1158,
  5381,
  4906,
  2825,
  3300,
  7361,
  1062,
  1385,
  131,
  2792,
  3300,
  1346,
  833,
  2145,
  2787,
  8024,
  6760,
  1355,
  2544,
  1300,
  1333,
  3152,
  8024,
  1131,
  955,
  6913,
  6435,
  1141,
  8024,
  1168,
  5031,
  6468,
  833,
  4385,
  1767,
  2544,
  1300,
  1277,
  7566,
  1357,
  511,
  137,
  1920,
  1039,
  3208,
  5763,
  689,
  2135,
  3175,
  2544,
  1300,
  137,
  8246,
  3249,
  3827,
  5381,
  137,
  756,
  1298,
  3995,
  1395,
  833,
  6379,
  3302,
  1218,
  3300,
  7361,
  1062,
  1385,
  137,
  756,
  3952,
  3180,
  6121,
  5381,
  137,
  3204,
  3209,
  1962,
  782,
  5357,
  2157,
  3124,
  137,
  3204,
  3209,
  5722,


## 本次微調需要得到正面/負面的判斷結果，因此挑選AutoModelForSequenceClassification
- 輸出結果為正面/負面，因此num_labels=2

In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-chinese", num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 利用TrainingArguments設定微調參數

In [27]:
from transformers import TrainingArguments
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(output_dir="test_trainer_chinese", evaluation_strategy="epoch")


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## 利用Trainer進行訓練
- 此處須輸入wandb key

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

## 利用pipeline進行測試
- LABEL_0：負面
- LABEL_1：正面

In [ ]:
from transformers import pipeline
pipe = pipeline("sentiment-analysis", model='test_trainer_chinese/checkpoint-1500', tokenizer=tokenizer)

In [ ]:
pipe("我喜歡這個產品")